# 0508 Error Compensation: 2D Experiment Suite (No MNIST/CIFAR)

This notebook implements and runs the 2D experiments described in:
`C:\\Users\\Xing\\Desktop\\0508 errorcompensation.md`.

Outputs:
- figures -> `figures/`
- results  -> `results/` (JSON summaries + `.npz` traces)


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE_DIR = Path("C:/Users/Xing/Desktop/error_compensation_2d_20260508_110807")
FIG_DIR = BASE_DIR / "figures"
RES_DIR = BASE_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


BASE_DIR: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807


In [2]:
@dataclass
class Quad2D:
    a: float
    b: float

    def value(self, x: np.ndarray) -> float:
        return 0.5 * (self.a * float(x[0] ** 2) + self.b * float(x[1] ** 2))

    def grad(self, x: np.ndarray) -> np.ndarray:
        return np.array([self.a * float(x[0]), self.b * float(x[1])], dtype=float)


def unit(theta: float) -> np.ndarray:
    return np.array([np.cos(theta), np.sin(theta)], dtype=float)


def project(g: np.ndarray, p: np.ndarray) -> np.ndarray:
    return p * float(np.dot(p, g))


In [3]:
def phi_positive_eps(r_abs: float, eps_a: float, eps_s: float) -> float:
    return float(r_abs) / ((float(r_abs) + float(eps_a)) * (float(r_abs) + float(eps_s)))


def theorem_checks(*, a: float, b: float, lr: float, eps_a: float, eps_s: float, x0: np.ndarray) -> dict:
    R = float(a) * abs(float(x0[0]))
    r_star = float(np.sqrt(float(eps_a) * float(eps_s)))
    r0 = min(R, r_star)
    M0 = phi_positive_eps(r0, eps_a, eps_s)

    ok1 = (lr * a) < (2.0 * eps_a)
    ok2 = (lr * b * M0) <= 0.5

    info = {
        "eta_a": lr * a,
        "two_eps_a": 2.0 * eps_a,
        "ok_eta_a_lt_2epsa": bool(ok1),
        "R": R,
        "r_star": r_star,
        "r0": r0,
        "M0": M0,
        "eta_b_M0": lr * b * M0,
        "ok_eta_b_M0_le_half": bool(ok2),
    }
    print("Theorem checks")
    print("--------------")
    print("eta*a < 2 eps_a:", info["eta_a"], "<", info["two_eps_a"], "=", info["ok_eta_a_lt_2epsa"])
    print("M0:", info["M0"])
    print("eta*b*M0 <= 1/2:", info["eta_b_M0"], "<=", 0.5, "=", info["ok_eta_b_M0_le_half"])
    return info


In [4]:
class Optim:
    name: str

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        raise NotImplementedError


class ProjSGD(Optim):
    def __init__(self, *, lr: float, angle_fn):
        self.name = "proj_sgd"
        self.lr = float(lr)
        self.angle_fn = angle_fn

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        u = project(g, p)
        x_next = x - self.lr * u
        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float("nan"),
            "s": float("nan"),
            "phi_eff": float("nan"),
            "rho": float("nan"),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float("nan"),
        }


class FiraScalar(Optim):
    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, s_clip: tuple[float, float] | None = None, limiter_gamma: float | None = None):
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.s_clip = s_clip
        self.limiter_gamma = limiter_gamma

        if limiter_gamma is not None:
            self.name = "fira_limiter"
        else:
            self.name = "fira_raw" if s_clip is None else "fira_clipped"

        self._prev_u_perp_norm = None

    def _psi(self, r: float) -> float:
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        if self.s_clip is None:
            s_t = float(s_raw)
        else:
            s_min, s_max = self.s_clip
            s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        u_perp = s_t * g_perp
        u_perp_eff = u_perp.copy()
        if self.limiter_gamma is not None:
            cur = float(np.linalg.norm(u_perp))
            if (
                self._prev_u_perp_norm is not None
                and self._prev_u_perp_norm > 0
                and cur > self.limiter_gamma * self._prev_u_perp_norm
            ):
                u_perp_eff = u_perp * (self.limiter_gamma * self._prev_u_perp_norm / cur)
                cur = float(np.linalg.norm(u_perp_eff))
            self._prev_u_perp_norm = cur

        u = u_parallel + u_perp_eff
        x_next = x - self.lr * u

        gperp_norm = float(np.linalg.norm(g_perp))
        phi_eff = float(np.linalg.norm(u_perp_eff) / (gperp_norm + 1e-12))

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "phi_eff": float(phi_eff),
            "rho": float("nan"),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float("nan"),
        }


class BECRFiraToy(Optim):
    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, rho: float, s_clip: tuple[float, float], limiter_gamma: float | None = None):
        self.name = "becr_fira_toy"
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.rho = float(rho)
        self.s_clip = s_clip
        self.limiter_gamma = limiter_gamma

        self.e = np.zeros((2,), dtype=float)  # raw-gradient units
        self._prev_u_perp_norm = None

    def _psi(self, r: float) -> float:
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        s_min, s_max = self.s_clip
        s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        h = g_perp + self.e
        z = self.rho * h
        z_eff = z.copy()

        u_perp = s_t * z_eff
        if self.limiter_gamma is not None:
            cur = float(np.linalg.norm(u_perp))
            if (
                self._prev_u_perp_norm is not None
                and self._prev_u_perp_norm > 0
                and cur > self.limiter_gamma * self._prev_u_perp_norm
            ):
                tau = self.limiter_gamma * self._prev_u_perp_norm / cur
                z_eff = tau * z
                u_perp = s_t * z_eff
                cur = float(np.linalg.norm(u_perp))
            self._prev_u_perp_norm = cur

        self.e = h - z_eff

        u = u_parallel + u_perp
        x_next = x - self.lr * u

        gperp_norm = float(np.linalg.norm(g_perp))
        phi_eff = float(np.linalg.norm(u_perp) / (gperp_norm + 1e-12))

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "phi_eff": float(phi_eff),
            "rho": float(self.rho),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float(np.linalg.norm(self.e)),
        }


class WrongUnitsNaiveEF(Optim):
    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, s_clip: tuple[float, float], rho: float):
        self.name = "wrong_units_naive_ef"
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.s_clip = s_clip
        self.rho = float(rho)

        self.e = np.zeros((2,), dtype=float)  # update-units (wrong)

    def _psi(self, r: float) -> float:
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        s_min, s_max = self.s_clip
        s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        h = self.e + self.lr * g_perp  # wrong units
        z = self.rho * h
        self.e = h - z
        u_perp = s_t * z

        u = u_parallel + u_perp
        x_next = x - self.lr * u

        gperp_norm = float(np.linalg.norm(g_perp))
        phi_eff = float(np.linalg.norm(u_perp) / (gperp_norm + 1e-12))

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "phi_eff": float(phi_eff),
            "rho": float(self.rho),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float(np.linalg.norm(self.e)),
        }


In [5]:
def run_experiment(*, exp_name: str, quad: Quad2D, angle_fn, optimizers: list[Optim], x0: np.ndarray, steps: int, noise_std: float = 0.0, seed: int = 0) -> dict:
    rng = np.random.default_rng(int(seed))
    traces = {}

    for opt in optimizers:
        x = np.array(x0, dtype=float)

        xs = np.zeros((steps + 1,), dtype=float)
        ys = np.zeros((steps + 1,), dtype=float)
        f = np.zeros((steps + 1,), dtype=float)
        grad_norm = np.zeros((steps + 1,), dtype=float)
        phi_raw = np.zeros((steps + 1,), dtype=float)
        s = np.zeros((steps + 1,), dtype=float)
        phi_eff = np.zeros((steps + 1,), dtype=float)
        e_norm = np.zeros((steps + 1,), dtype=float)
        theta = np.zeros((steps + 1,), dtype=float)

        xs[0], ys[0] = float(x[0]), float(x[1])
        f[0] = quad.value(x)
        grad_norm[0] = float(np.linalg.norm(quad.grad(x)))
        phi_raw[0] = float("nan")
        s[0] = float("nan")
        phi_eff[0] = float("nan")
        e_norm[0] = float("nan")
        theta[0] = float(angle_fn(0))

        for t in range(steps):
            g = quad.grad(x)
            if noise_std > 0:
                g = g + float(noise_std) * rng.standard_normal((2,))
            x, info = opt.step(x, g, t)
            xs[t + 1], ys[t + 1] = float(info["x"]), float(info["y"])
            f[t + 1] = quad.value(x)
            grad_norm[t + 1] = float(np.linalg.norm(quad.grad(x)))
            phi_raw[t + 1] = float(info.get("phi_raw", float("nan")))
            s[t + 1] = float(info.get("s", float("nan")))
            phi_eff[t + 1] = float(info.get("phi_eff", float("nan")))
            e_norm[t + 1] = float(info.get("e_norm", float("nan")))
            theta[t + 1] = float(info.get("theta", float(angle_fn(t))))

        traces[opt.name] = {
            "x": xs,
            "y": ys,
            "f": f,
            "grad_norm": grad_norm,
            "phi_raw": phi_raw,
            "s": s,
            "phi_eff": phi_eff,
            "phi_cum": np.nancumsum(phi_raw),
            "s_cum": np.nancumsum(s),
            "e_norm": e_norm,
            "theta": theta,
        }

    # Save raw traces
    npz_payload = {}
    for name, tr in traces.items():
        for k, v in tr.items():
            npz_payload[f"{name}__{k}"] = np.asarray(v)
    np.savez_compressed(RES_DIR / f"{exp_name}__traces.npz", **npz_payload)

    # Common per-series plots
    def _plot_one(key: str, ylabel: str, out_name: str, *, yscale: str | None = None):
        fig, ax = plt.subplots(figsize=(6.2, 4.0))
        for name, tr in traces.items():
            ax.plot(tr[key], linewidth=1.5, label=name)
        ax.set_title(f"{exp_name}: {out_name}")
        ax.set_xlabel("step")
        ax.set_ylabel(ylabel)
        if yscale is not None:
            ax.set_yscale(yscale)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{exp_name}__{out_name}.png", dpi=180)
        plt.close(fig)

    _plot_one("x", "x_t", "x")
    _plot_one("y", "y_t", "y")
    _plot_one("grad_norm", "||grad||", "grad_norm", yscale="log")
    _plot_one("f", "f(x_t)", "objective", yscale="log")
    _plot_one("phi_raw", "phi_raw", "phi_raw")
    _plot_one("phi_cum", "cum(phi_raw)", "phi_cum")
    _plot_one("s", "s_t", "s")
    _plot_one("s_cum", "cum(s)", "s_cum")
    _plot_one("phi_eff", "phi_eff", "phi_eff")
    _plot_one("e_norm", "||e_t||", "e_norm")

    # Paper-style composite (Figure 1-style): 6 panels
    label_map = {
        "proj_sgd": "ProjSGD",
        "fira_raw": "Fira (raw)",
        "fira_clipped": "Fira (clipped)",
        "fira_limiter": "Fira (limiter)",
        "becr_fira_toy": "BECR-Fira",
        "wrong_units_naive_ef": "Naive EF (wrong units)",
    }
    color_map = {
        "proj_sgd": "#666666",
        "fira_raw": "#d62728",
        "fira_clipped": "#1f77b4",
        "fira_limiter": "#9467bd",
        "becr_fira_toy": "#2ca02c",
        "wrong_units_naive_ef": "#ff7f0e",
    }
    order = ["proj_sgd", "fira_raw", "fira_clipped", "fira_limiter", "becr_fira_toy", "wrong_units_naive_ef"]

    def _applied_scale(name: str, tr: dict) -> np.ndarray:
        if name == "fira_limiter":
            return np.asarray(tr["phi_eff"], dtype=float)
        if name in {"fira_clipped", "becr_fira_toy", "wrong_units_naive_ef"}:
            return np.asarray(tr["s"], dtype=float)
        if name == "fira_raw":
            return np.asarray(tr["phi_raw"], dtype=float)
        return np.asarray(tr["s"], dtype=float)

    def _cumsum_nan0(x: np.ndarray) -> np.ndarray:
        return np.cumsum(np.nan_to_num(np.asarray(x, dtype=float), nan=0.0))

    fig, axs = plt.subplots(2, 3, figsize=(15.2, 7.6))
    ax_gn, ax_y, ax_scale = axs[0, 0], axs[0, 1], axs[0, 2]
    ax_cum, ax_e, ax_obj = axs[1, 0], axs[1, 1], axs[1, 2]

    for name in order:
        if name not in traces:
            continue
        tr = traces[name]
        label = label_map.get(name, name)
        color = color_map.get(name, None)
        linestyle = "--" if name == "wrong_units_naive_ef" else "-"

        # (a) gradient norm
        ax_gn.plot(tr["grad_norm"], linewidth=1.7, color=color, linestyle=linestyle, label=label)

        # (b) |y_t|
        y_abs = np.maximum(np.abs(tr["y"]), 1e-300)
        ax_y.plot(y_abs, linewidth=1.6, color=color, linestyle=linestyle)

        # (c) raw phi_t vs clipped/applied scale s_t (applied scale depends on method)
        s_used = _applied_scale(name, tr)
        s_used_pos = np.where(np.isfinite(s_used) & (s_used > 0), s_used, np.nan)
        if np.isfinite(s_used_pos).any():
            ax_scale.plot(s_used_pos, linewidth=1.4, color=color, linestyle=linestyle)

        # (d) cumulative sum of applied scale
        s_cum_used = _cumsum_nan0(s_used)
        s_cum_pos = np.where(s_cum_used > 0, s_cum_used, np.nan)
        if np.isfinite(s_cum_pos).any():
            ax_cum.plot(s_cum_pos, linewidth=1.4, color=color, linestyle=linestyle)

        # (e) residual norm
        e_series = np.asarray(tr["e_norm"], dtype=float)
        e_pos = np.where(np.isfinite(e_series) & (e_series > 0), e_series, np.nan)
        if np.isfinite(e_pos).any():
            ax_e.plot(e_pos, linewidth=1.4, color=color, linestyle=linestyle)

        # (f) objective
        f_series = np.maximum(np.asarray(tr["f"], dtype=float), 1e-300)
        ax_obj.plot(f_series, linewidth=1.6, color=color, linestyle=linestyle)

    ax_gn.set_title("(a) ||∇f(x_t,y_t)||")
    ax_gn.set_xlabel("step")
    ax_gn.set_ylabel("||grad||")
    ax_gn.set_yscale("log")
    ax_gn.grid(True, alpha=0.3)
    ax_gn.legend(loc="best", fontsize=9)

    ax_y.set_title("(b) |y_t|")
    ax_y.set_xlabel("step")
    ax_y.set_ylabel("|y|")
    ax_y.set_yscale("log")
    ax_y.grid(True, alpha=0.3)

    ax_scale.set_title("(c) raw φ_t vs clipped/applied s_t")
    ax_scale.set_xlabel("step")
    ax_scale.set_ylabel("scale")
    ax_scale.set_yscale("log")
    ax_scale.grid(True, alpha=0.3)

    ax_cum.set_title("(d) cumulative ∑ scale")
    ax_cum.set_xlabel("step")
    ax_cum.set_ylabel("∑ scale")
    ax_cum.set_yscale("log")
    ax_cum.grid(True, alpha=0.3)

    ax_e.set_title("(e) residual norm ||e_t||")
    ax_e.set_xlabel("step")
    ax_e.set_ylabel("||e||")
    ax_e.set_yscale("log")
    ax_e.grid(True, alpha=0.3)

    ax_obj.set_title("(f) objective f(x_t,y_t)")
    ax_obj.set_xlabel("step")
    ax_obj.set_ylabel("f")
    ax_obj.set_yscale("log")
    ax_obj.grid(True, alpha=0.3)

    fig.suptitle(exp_name)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(FIG_DIR / f"{exp_name}__FIG1_MAIN.png", dpi=240)
    fig.savefig(FIG_DIR / f"{exp_name}__FIG1_MAIN.pdf")
    # Backward-compat name
    fig.savefig(FIG_DIR / f"{exp_name}__PAPER_MAIN.png", dpi=240)
    fig.savefig(FIG_DIR / f"{exp_name}__PAPER_MAIN.pdf")
    plt.close(fig)

    # Summary table
    summary = {
        "exp_name": exp_name,
        "quad": {"a": quad.a, "b": quad.b},
        "steps": int(steps),
        "noise_std": float(noise_std),
        "seed": int(seed),
        "optimizers": {},
    }

    for name, tr in traces.items():
        summary["optimizers"][name] = {
            "x_final": float(tr["x"][-1]),
            "y_final": float(tr["y"][-1]),
            "grad_norm_final": float(tr["grad_norm"][-1]),
            "residual_norm_final": float(tr["e_norm"][-1]) if np.isfinite(tr["e_norm"][-1]) else None,
            "sum_phi_raw": float(tr["phi_cum"][-1]) if np.isfinite(tr["phi_cum"][-1]) else None,
            "sum_s": float(tr["s_cum"][-1]) if np.isfinite(tr["s_cum"][-1]) else None,
        }

    (RES_DIR / f"{exp_name}__summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


In [6]:
def fixed_angle(theta: float):
    return lambda t: float(theta)


def step_angle(theta0: float, theta1: float, switch_t: int):
    def f(t: int) -> float:
        return float(theta0) if int(t) < int(switch_t) else float(theta1)
    return f


def rotating_angle(theta0: float, omega: float):
    return lambda t: float(theta0 + omega * float(t))


summaries = []

# (A) Theorem-regime fixed subspace (P=e1)
quadA = Quad2D(a=1.0, b=1.0)
lrA = 0.5
eps_aA = 1.0
eps_sA = 1.0
x0A = np.array([1.0, 1.0], dtype=float)
angleA = fixed_angle(0.0)

checksA = theorem_checks(a=quadA.a, b=quadA.b, lr=lrA, eps_a=eps_aA, eps_s=eps_sA, x0=x0A)
(RES_DIR / "theorem_regime_fixed_subspace__THEOREM_CHECKS.json").write_text(json.dumps(checksA, indent=2), encoding="utf-8")

optsA = [
    ProjSGD(lr=lrA, angle_fn=angleA),
    FiraScalar(lr=lrA, angle_fn=angleA, eps_a=eps_aA, eps_s=eps_sA, s_clip=None),
    FiraScalar(lr=lrA, angle_fn=angleA, eps_a=eps_aA, eps_s=eps_sA, s_clip=(0.2, 0.5)),
    BECRFiraToy(lr=lrA, angle_fn=angleA, eps_a=eps_aA, eps_s=eps_sA, rho=0.5, s_clip=(0.2, 0.5), limiter_gamma=None),
    WrongUnitsNaiveEF(lr=lrA, angle_fn=angleA, eps_a=eps_aA, eps_s=eps_sA, s_clip=(0.2, 0.5), rho=0.5),
]

summaries.append(
    run_experiment(
        exp_name="theorem_regime_fixed_subspace",
        quad=quadA,
        angle_fn=angleA,
        optimizers=optsA,
        x0=x0A,
        steps=600,
        noise_std=0.0,
        seed=0,
    )
)

# (B) stale_then_switch (same safe eps/lr to avoid projected 2-cycle)
angleB = step_angle(0.0, np.pi / 2.0, switch_t=200)
optsB = [
    ProjSGD(lr=lrA, angle_fn=angleB),
    FiraScalar(lr=lrA, angle_fn=angleB, eps_a=eps_aA, eps_s=eps_sA, s_clip=None),
    FiraScalar(lr=lrA, angle_fn=angleB, eps_a=eps_aA, eps_s=eps_sA, s_clip=(0.2, 0.5)),
    BECRFiraToy(lr=lrA, angle_fn=angleB, eps_a=eps_aA, eps_s=eps_sA, rho=0.5, s_clip=(0.2, 0.5), limiter_gamma=None),
    WrongUnitsNaiveEF(lr=lrA, angle_fn=angleB, eps_a=eps_aA, eps_s=eps_sA, s_clip=(0.2, 0.5), rho=0.5),
]

summaries.append(
    run_experiment(
        exp_name="stale_then_switch_safe",
        quad=quadA,
        angle_fn=angleB,
        optimizers=optsB,
        x0=x0A,
        steps=600,
        noise_std=0.0,
        seed=0,
    )
)

# (C) rotating_noise stress test (matches 0508 note settings)
quadC = Quad2D(a=1.0, b=5.0)
lrC = 0.05
eps_aC = 1e-3
eps_sC = 1e-3
x0C = np.array([1.0, 1.0], dtype=float)
angleC = rotating_angle(0.0, omega=0.02)

optsC = [
    ProjSGD(lr=lrC, angle_fn=angleC),
    FiraScalar(lr=lrC, angle_fn=angleC, eps_a=eps_aC, eps_s=eps_sC, s_clip=None),
    FiraScalar(lr=lrC, angle_fn=angleC, eps_a=eps_aC, eps_s=eps_sC, s_clip=(0.1, 10.0)),
    FiraScalar(lr=lrC, angle_fn=angleC, eps_a=eps_aC, eps_s=eps_sC, s_clip=None, limiter_gamma=2.0),
    BECRFiraToy(lr=lrC, angle_fn=angleC, eps_a=eps_aC, eps_s=eps_sC, rho=0.5, s_clip=(0.1, 10.0), limiter_gamma=None),
    WrongUnitsNaiveEF(lr=lrC, angle_fn=angleC, eps_a=eps_aC, eps_s=eps_sC, s_clip=(0.1, 10.0), rho=0.5),
]

summaries.append(
    run_experiment(
        exp_name="rotating_noise_stress",
        quad=quadC,
        angle_fn=angleC,
        optimizers=optsC,
        x0=x0C,
        steps=800,
        noise_std=0.05,
        seed=0,
    )
)

(RES_DIR / "0508_SUITE__ALL_SUMMARIES.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")
print("Done. Wrote figures to:", FIG_DIR)
print("Done. Wrote results to:", RES_DIR)


Theorem checks
--------------
eta*a < 2 eps_a: 0.5 < 2.0 = True
M0: 0.25
eta*b*M0 <= 1/2: 0.125 <= 0.5 = True


Done. Wrote figures to: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807\figures
Done. Wrote results to: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807\results
